# A verified 89-qubit inverse square root, in 1,466 Toffolis

**Google Quantum AI — Verifiable Quantum Circuit Compilation (Benchmark I: inverse square root)**

This notebook verifies one circuit and writes the artifact to submit. It does not train a
model, sample a search space, or estimate anything. The circuit either passes the official
verifier on every shot or it does not, and the receipt below is the whole argument.

| | |
|---|---|
| logical qubits `Q` | **89** |
| max Toffoli-like | **1,466** |
| worst error / tolerance | **0.9996** (passes with 0.04% of the tolerance left) |
| artifact sha256 | `ce04c9d5…4e58fa9de` |
| official verifier verdict | **PASS** |

The rest of this notebook is: what is being scored (below), the arithmetic that produced the
circuit (§1), the certification (§2), and the four ideas that were built, measured and
rejected because they came out worse (§8). The rejected ones matter as much as the accepted
one, because each was a plausible 19% saving that turned out to be a loss.

*The circuit is a 245 KB compiled `.kmb`. In this notebook it comes from an attached dataset;
`kaggle/README.md` in the source repository explains how to attach it.*


## What is being scored

Benchmark I asks for a reversible circuit that computes `1/sqrt(x)` on a superposition, with
the input register restored, every scratch qubit returned to `|0⟩`, and the right phase on
every shot. The cost is

```
cost = Q x (Toffoli-equivalents + T/4)
```

against a reference cost of **2,076,774.23**. One number, and its two factors pull against each
other: at this operating point one qubit is worth about **19.9 Toffolis**. Any change that
trades qubits for gates therefore has to be priced before it is built, not after.

The portfolio score is the geometric mean of the three per-benchmark ratios,
`(ref_i / candidate_i)^(1/3)`. Benchmark I alone can therefore lift the aggregate only to
`(2076774.23 / 157601.08)^(1/3) ~= 2.36x`, because the other two benchmarks would sit at their
reference cost.


## 1. The approach

### 1.1 Normalise with a leading-zero counter, not a division

`1/sqrt(x)` needs `x` in a narrow range before any polynomial is accurate. Rather than divide,
the circuit counts leading zeros with a prefix-AND chain — `chain[m] = [top m+2 bits are zero]`
— and derives the shift class from three bits of that count (`f4`, `f3`, `f2`) with
`chain[4j-2]` taps. Nineteen Toffolis replace the `15 + 7 + 3` an independent three-stage
counter needed, because the nested prefixes make the three bits fall out of one chain.

The shift itself is five `cswap` stages of 16, 8, 4, 2 and 1, each conditional on one bit of
the class, and each undone on the way out by the same stage in reverse order.

### 1.2 Evaluate the polynomial with a folded in-place adder

The Horner evaluation and the final subtraction both reduce to "add a shorter addend into a
wider target". The adder used is the folded (Gidney) form: `n - 1` Toffolis for an `n`-bit
addition, with the carry chain uncomputed by measurement-and-reset of the intermediate
carries rather than by a second Toffoli pass. That is the single most load-bearing choice in
the design — a textbook `n - 1` ripple adder costs `2n - 2` here, and §4 prices what that costs.

### 1.3 Share the decision diagrams

The phase tables are multi-output Boolean functions, so they are synthesised as PDDDs. The
tables share most of their structure, and synthesising them *jointly* rather than one at a
time is worth two nodes: 26 against 28 for the independent plans. The champion is an
11-node coarse stage plus a 28-node curve stage covering curves 4 to 6, shared across all
outputs of the same input.

### 1.4 Calibrate phases, then remove what is already zero

The phase angles are calibrated against the verifier's own tolerance. The calibration table
holds 48 deltas; three are exactly zero and one is redundant, and the shipped rewrite removes
four of them, leaving **44 rotations**. Naming the zero entries again does *not* save three
more: the compiler leaves 45 `z_pow` ops, and re-listing indices 15-17 shifts the index
mapping and deletes three real rotations, which fails at `x = 1` with a 0.125-turn phase
error against a tolerance of 5.96e-8. 44 is the floor, and §4 records how that was checked.


## 2. Set up

The competition bundle (the pinned `kickmix` extension plus the untouched official verifiers)
is mounted read-only at `/kaggle/input`. The first cell locates it and refuses to continue
without it, because a verifier that silently is not there would turn this notebook into a
resource estimate, and a resource estimate is not a result.

The circuit arrives as a second input, a 245 KB dataset. It has to: competition notebooks have
no internet and no access to the repository it was built in.


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

CANDIDATE_ROOTS = [
    Path("/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation"),
    Path("/kaggle/input/google-verifiable-quantum-circuit-compilation"),
    Path(os.environ.get("QCOMP_ROOT", "/nonexistent")),
]

ROOT = next(
    (p for p in CANDIDATE_ROOTS if (p / "verifiers").is_dir() and (p / "kickmix").is_dir()),
    None,
)
if ROOT is None:
    seen = sorted(str(p) for p in Path("/kaggle/input").glob("**/kickmix")) if Path("/kaggle/input").exists() else []
    raise SystemExit(
        "Competition bundle not found. Attach the competition data, or set QCOMP_ROOT.\n"
        f"Searched: {[str(p) for p in CANDIDATE_ROOTS]}\n"
        f"kickmix directories visible: {seen}"
    )

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'kickmix'))  # the compiled .so lives one level down
import kickmix as km  # noqa: E402  (must follow the path setup)

VERIFIER = ROOT / "verifiers" / "invsqrt" / "verify_submission_invsqrt.py"
assert VERIFIER.is_file(), f"official verifier missing at {VERIFIER}"

print("competition root :", ROOT)
print("kickmix          :", getattr(km, "__version__", "version not exposed"), "->", km.__file__)
print("verifier         :", VERIFIER.name, f"({VERIFIER.stat().st_size:,} bytes)")
print("python           :", sys.version.split()[0])


## 3. The circuit, and a hash check before anything is claimed

A circuit file is only evidence if it is *this* circuit. The sha256 below is the value pinned
by the champion's rebuild test (`tests/test_dyn_invsqrt_champion_rebuild.py`), so a wrong or
truncated download fails here, loudly, instead of producing a plausible-looking receipt for a
different circuit.


In [ ]:
PINNED_CIRCUIT_SHA256 = "ce04c9d54dc42e42455444022b6652a6efb0be2630aca7b485d3efd4e58fa9de"
PINNED_QUBITS = 89
PINNED_MAX_TOFFOLI = 1466
CIRCUIT_NAME = "invsqrt_89q_shared_pddd.kmb"


def find_circuit() -> Path | None:
    for base in (Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()):
        if not base.exists():
            continue
        for candidate in base.rglob(CIRCUIT_NAME):
            return candidate
    return None


CIRCUIT = find_circuit()
if CIRCUIT is None:
    raise SystemExit(
        f"{CIRCUIT_NAME} is not attached. Add the 245 KB circuit as a Kaggle dataset input; "
        "see kaggle/README.md in the source repository."
    )

digest = hashlib.sha256(CIRCUIT.read_bytes()).hexdigest()
print("circuit :", CIRCUIT)
print("bytes   :", f"{CIRCUIT.stat().st_size:,}")
print("sha256  :", digest)
assert digest == PINNED_CIRCUIT_SHA256, (
    "this is not the pinned champion artifact:\n"
    f"  expected {PINNED_CIRCUIT_SHA256}\n"
    f"  got      {digest}"
)
print("\nhash matches the pinned champion artifact")


## 4. Run the official verifier

No reimplementation, no shortcut. The untouched `verify_submission_invsqrt.py` simulates the
circuit on many inputs and checks every shot: the input register restored, every scratch
qubit back to `|0⟩`, and the phase within tolerance.

`SHOTS_PER_X` is `None`, which runs the verifier's own budget rather than a reduced sample:
64 shots per input, 200,540,928 in total. That is the graded configuration, so the receipt in
§6 is the one the score comes from. A smaller value is legal only in multiples of 64 (the
verifier works in 256-shot batches, and 3,133,452 = 4 x 783,363), and it changes the sample,
not the pass/fail logic.


In [ ]:
SHOTS_PER_X = None  # None = the verifier's default, 64 per x (200,540,928 shots): the graded run

cmd = [sys.executable, str(VERIFIER), str(CIRCUIT)]
if SHOTS_PER_X is not None:
    cmd += ["--shots-per-x", str(SHOTS_PER_X)]

print("$", " ".join(cmd), "\n")
env = dict(os.environ)
env['PYTHONPATH'] = os.pathsep.join([str(ROOT / 'kickmix'), env.get('PYTHONPATH', '')])
proc = subprocess.run(cmd, capture_output=True, text=True, timeout=60 * 120, env=env)

if proc.returncode != 0:
    print(proc.stdout[-6000:])
    print(proc.stderr[-6000:], file=sys.stderr)
    raise SystemExit(f"verifier exited {proc.returncode}")

start = proc.stdout.find("{")
receipt = json.loads(proc.stdout[start:])
print(json.dumps(receipt, indent=2, sort_keys=True)[:6000])


In [ ]:
status = receipt.get("status")
assert status == "PASS", f"verifier verdict is {status!r}, not PASS: {receipt}"

# Deterministic fields: these identify the circuit and the design, so they must match exactly.
assert receipt.get("circuit_sha256") == PINNED_CIRCUIT_SHA256, "receipt is for a different circuit"
assert receipt.get("num_qubits") == PINNED_QUBITS, receipt.get("num_qubits")
assert receipt.get("max_toffoli_count") == PINNED_MAX_TOFFOLI, receipt.get("max_toffoli_count")

# Sampled fields: these move with the shot budget, so they are bounded, not pinned.
err = receipt.get("worst_error_over_tolerance")
assert err is None or err <= 1.0, f"worst error/tolerance is {err}, above the 1.0 failure line"

print("verdict             :", status)
print("verified shots      :", f"{receipt.get('verified_shots', 0):,}")
print("worst err/tolerance :", f"{err:.6f} of 1.0" if err is not None else "not reported")
print("receipt fields      :", ", ".join(sorted(receipt)))


## 5. Price it

The cost is `Q x toffoli-equivalent`, where the toffoli-equivalent already carries the `T/4`
weight. The receipt reports both `score` and `toffoli_equivalent`, so the cell recomputes the
product and refuses to print a ratio if they disagree — a score that does not equal
`Q x toffoli-equivalent` is not a score, it is a number in a JSON file.

The pinned reference figure, `157,601.0808`, came from a 200,540,928-shot run. A run at the
smaller budget here will land in a narrow band around it rather than on it, because the mean
Toffoli count is a sample mean.


In [ ]:
REFERENCE_COST = 2_076_774.233715282   # pinned reference implementation
PINNED_SCORE = 157_601.0808             # champion, 200,540,928 shots

qubits = int(receipt.get("num_qubits") or PINNED_QUBITS)
score = float(receipt.get("score") or 0.0)
equiv = float(receipt.get("toffoli_equivalent") or 0.0)

if score <= 0.0 or equiv <= 0.0:
    raise SystemExit(f"receipt has no score/toffoli_equivalent to price: {sorted(receipt)}")

recomputed = qubits * equiv
assert abs(recomputed - score) <= 1e-9 * score, (
    f"score {score!r} is not Q x toffoli_equivalent ({recomputed!r})"
)

ratio = REFERENCE_COST / score
print(f"qubits              : {qubits}")
print(f"toffoli-equivalent  : {equiv:,.2f}  (T/4 weight included)")
print(f"score (Q x equiv)   : {score:,.4f}")
print(f"reference cost      : {REFERENCE_COST:,.2f}")
print(f"improvement         : {ratio:.4f}x on Benchmark I")
print(f"portfolio effect    : {ratio ** (1 / 3):.4f}x  (geometric mean, other two at reference)")
print(f"pinned reference run: {PINNED_SCORE:,.4f}  (200,540,928 shots)")
print(f"drift vs pinned     : {score - PINNED_SCORE:+,.4f}")


## 6. Write the submission artifact

A receipt is evidence that a circuit was verified; it is not the submission. The `.kmb` is.
Both are written to `/kaggle/working/submission/` so the notebook output carries the circuit and
the proof together.

The competition's Submit page decides whether a file submission or a notebook version is
expected — confirm which before the graded run, and use the filename it names. Nothing here
renames or re-encodes the circuit: the bytes are copied unchanged, so the submitted artifact
hashes to the same `ce04c9d5…`.


In [ ]:
OUT = Path("/kaggle/working/submission")
OUT.mkdir(parents=True, exist_ok=True)

submitted_circuit = OUT / "invsqrt.kmb"
shutil.copyfile(CIRCUIT, submitted_circuit)
assert hashlib.sha256(submitted_circuit.read_bytes()).hexdigest() == PINNED_CIRCUIT_SHA256

(OUT / "receipt.json").write_text(json.dumps(receipt, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(OUT / "summary.json").write_text(
    json.dumps(
        {
            "benchmark": "invsqrt",
            "circuit_sha256": PINNED_CIRCUIT_SHA256,
            "num_qubits": qubits,
            "max_toffoli_count": receipt.get("max_toffoli_count"),
            "toffoli_equivalent": equiv,
            "score": score,
            "reference_cost": REFERENCE_COST,
            "improvement_factor": ratio,
            "verifier_status": status,
            "verified_shots": receipt.get("verified_shots"),
            "worst_error_over_tolerance": receipt.get("worst_error_over_tolerance"),
            "shots_per_x": SHOTS_PER_X,
        },
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

for path in sorted(OUT.iterdir()):
    print(f"{path.stat().st_size:>10,}  {path.name}")


## 7. Results

| quantity | value | source |
|---|---|---|
| logical qubits `Q` | **89** | verifier |
| max Toffoli-like | **1,466** | verifier |
| toffoli-equivalent | **1,770.8** | verifier |
| score, `Q x equivalent` | **157,601.08** | verifier, 200,540,928 shots |
| worst error / tolerance | **0.999623** | verifier |
| rotations after calibration | 44 | build |
| artifact sha256 | `ce04c9d5…4e58fa9de` | pinned |
| verdict | **PASS** | verifier |

`0.999623` is the number that matters. It is the worst ratio of measured error to the tolerance
the verifier allows, across every shot and every input in the sample — the circuit passes with
0.04% of the budget left. That is not a comfortable margin, and it is why the rejected ideas
below are rejected rather than queued: the tolerance is essentially spent, so anything that
perturbs the arithmetic further is a failing change, not a faster one.

## 8. What was tried and rejected

Every one of these was a plausible saving. Each was priced or built before being rejected, and
the rejection is part of the result.

**88 qubits by recycling carries with measurement-reset.** The carry chain is the peak, so
measuring and resetting carries would drop the peak by about eight qubits. It needs an `MR`
opcode, and the pinned competition build has none: its parser rejects `MR q0 b0` with "unknown
operation name", while `R` and `HMR` parse fine. A circuit the official verifier cannot read
cannot produce a receipt, and an unverifiable 19% is worth zero.

**A one-ancilla (Cuccaro) adder.** `2n - 2` Toffolis instead of `n - 1`, in exchange for
replacing the carry ancillas with one. Measured on the champion's own 46 additions: 758
Toffolis become 1,516. The qubit saving is 13 rather than 18, because a zero addend bit still
needs a real `|0>` control, so 13 x 19.9 = 259 gained against 758 lost.

**Dropping the zero-angle rotations.** Indices 15-17 of the calibration table are exactly `0.0`,
which looks like three free rotations. Rebuilding with them removed fails at `x = 1` with a
0.125-turn phase error, against a tolerance of 5.96e-8. The table has 48 entries and the
compiler emits 45 `z_pow` ops, so the indices do not line up the way the omission assumes: the
shipped rewrite already removes three zeros plus index 18, and re-naming 15-17 deletes three
*real* rotations.

**Dropping a correction row.** The polynomial correction is 8 signed-digit rows, so dropping one
looks like a free saving. Every single-row drop fails: evaluating the circuit's own integer model
over all 3,133,452 supported inputs against the verifier's target and tolerance puts the worst
error between 1.14x tolerance (row 6) and 48x (row 13), and even the best drop is wrong on 110 of
the inputs. A row is one signed digit of the `C2 x delta` product, so removing it is not a
rounding change, and it buys no width either: the residual stays 18 bits. Re-calibrating the
coefficient tables against a 7-row model puts row 6 at 0.98 of tolerance, which is inside the
3.7% by which that model under-reports this circuit's measured 0.99962 — a build question, not a
pass. Even if it passed, the 8 rows together are 132 Toffolis, about 9% of the circuit, so the
prize is 1.1% of the score.

**Why 89 is the floor for this design.** Every position of the wide addition needs a carry
while it is in flight, and the only measurements that let you recycle those carries are the
ones the pinned build cannot parse. Chunking the addition does not dodge it either: splitting
the target requires the boundary carry to survive into the next chunk, and an in-place adder
destroys the information needed to uncompute it — which is why the one-ancilla version has to
pay double in gates.

**The 62-qubit bar is priced in a separate record.** The score is `Q x toffoli-equivalent`, so
at `Q = 62` the bar is `te < 1,622.09` against this circuit's 1,770.8: width has to be bought
with gates, not given away. `docs/INVSQRT_SUB62_DESIGN.md` prices that exchange from the
allocation ladder — the 71 named registers plus the 18 carries of the wide adder, the segmented
build that aborts at 76 qubits on `Scratch function is not zero`, record-restoring the registers
to `Q ~ 60` at 130,260, and the digit-serial schedule at 34,997 qubits. None of the three yields a
better submission, which is why this notebook ships 89 and says so rather than claiming a width
it cannot reach.


## 9. Reproducing this from source

The circuit is generated, not hand-written. The whole path in the source repository is:

```
scripts/test_shared_pddd_circuit.py         # builds the circuit and writes the .kmb
tests/test_dyn_invsqrt_champion_rebuild.py  # rebuilds it and pins the hash above
```

The rebuild is a regression test, not a claim: it asserts the circuit hash, `Q = 89`, the max
Toffoli count, the worst error ratio and the shot count, and it fails if the build gets ten
times slower. It needs the `kickmix` extension, which targets CPython 3.13, so it is opt-in:

```bash
APEDP_Q_KICKMIX_PYTHON=/path/to/python3.13 pytest tests/test_dyn_invsqrt_champion_rebuild.py
```

The circuit itself lands in `artifacts/`, which is gitignored — the bytes are not in version
control, only the hash that identifies them and the receipt that certifies them.

**What is verified here versus what is not.** Benchmark I is verified end to end: a compiled
artifact, a hash, an official PASS, and a receipt. Benchmarks II and III are not submitted.
The Benchmark II decoder is a complete circuit whose counts come from the verifier's own
counting function, but it cannot be compiled against the pinned binary — its ledger clears are
measurement-and-reset operations and the build has no `MR` opcode — so it can be measured
locally and never certified. Benchmark III needs no new opcode either — its submission is
already `kickmix` circuits — and the width wall turned out to be solvable: a `brot = 10`,
`J = 10` family of 8 circuits fits **123 qubits** in **63,584 Toffolis** across both passes, and
the bundle's own `recover_words_from_circuits` recovers all 298,200 angle words exactly. What
remains is precision, and that is measured rather than argued. With the pinned FCIDUMP
mounted, the competition's `verify.py` returns **242.3 mHa** against a **20 mHa** tolerance for
that family — 338.3 mHa at 9 bits, 568.2 mHa at 8 bits — because a 12-bit table rounded down is
outside the error budget however exactly the words are loaded. So Benchmark III is absent
because it fails the verifier, not because it cannot be built; the earlier version of this
section said the opposite and was wrong on both counts.

## Files

| file | what it is |
|---|---|
| `invsqrt_89q_shared_pddd.kmb` | the compiled circuit, 245,010 bytes |
| `submission/receipt.json` | the official verifier's output, verbatim |
| `submission/summary.json` | the counts and the cost arithmetic |
